# 3일차 — Advanced Actor-Critic Methods

2026-07-29 (수) · 연속 행동 공간을 다루는 DDPG부터 최대 엔트로피 계열의 SAC·TAC까지, 최신 심층강화학습 알고리즘을 구현합니다.

> 위에서부터 순서대로 실행하세요. 뒤 교시가 앞 교시의 변수·클래스를 그대로 이어 씁니다.


## (이어받기) 2일차 1교시 — DQN 소개

오늘 코드가 이 정의를 그대로 이어 씁니다. **가장 먼저 한 번 실행**하세요.


In [ ]:
import random
from collections import deque
import numpy as np
import torch

class ReplayBuffer:
    """경험 재현 버퍼 — DQN, DDPG, SAC 3일 내내 재사용합니다"""
    def __init__(self, capacity=100_000, action_dtype=torch.int64):
        # action_dtype: 오늘 DQN은 행동이 "몇 번 행동"인 정수라 int64입니다.
        # 3일차 DDPG·SAC는 행동이 연속값(실수 벡터)이므로 float32로 바꿔 씁니다
        #   buffer = ReplayBuffer(100_000, action_dtype=torch.float32)
        # int64로 두면 실수 행동이 정수로 잘려 학습이 통째로 망가집니다.
        self.buffer = deque(maxlen=capacity)
        self.action_dtype = action_dtype

    def push(self, s, a, r, s_next, done):
        self.buffer.append((s, a, r, s_next, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        s, a, r, s_next, done = zip(*batch)
        # np.array로 한 번 묶고 텐서로 바꿉니다.
        # 배열 리스트를 텐서로 바로 만들면 파이토치가 하나씩 복사해 매우 느립니다.
        return (torch.as_tensor(np.array(s), dtype=torch.float32),
                torch.as_tensor(np.array(a), dtype=self.action_dtype),
                torch.as_tensor(np.array(r), dtype=torch.float32),
                torch.as_tensor(np.array(s_next), dtype=torch.float32),
                torch.as_tensor(np.array(done), dtype=torch.float32))

    def __len__(self):
        return len(self.buffer)

## 1교시 · DDPG 소개

`09:30 ~ 10:30` · `ddpg_networks.py`

- 연속 행동 공간에서 DQN이 동작하지 않는 이유를 이해한다
- 결정적 정책 경사와 DDPG의 4개 네트워크 구조를 설명할 수 있다


In [ ]:
import torch
import torch.nn as nn

class Actor(nn.Module):
    """상태 → 연속 행동 (결정적)"""
    def __init__(self, state_dim, action_dim, max_action):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 256), nn.ReLU(),
            nn.Linear(256, 256), nn.ReLU(),
            nn.Linear(256, action_dim), nn.Tanh(),   # [-1, 1]
        )
        self.max_action = max_action

    def forward(self, s):
        return self.net(s) * self.max_action         # 행동 범위로 스케일

class Critic(nn.Module):
    """(상태, 행동) → Q값 — 행동을 입력으로 받는 점이 DQN과 다름"""
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim + action_dim, 256), nn.ReLU(),
            nn.Linear(256, 256), nn.ReLU(),
            nn.Linear(256, 1),
        )

    def forward(self, s, a):
        return self.net(torch.cat([s, a], dim=-1)).squeeze(-1)

def soft_update(target, source, tau=0.005):
    """타깃 네트워크 소프트 업데이트 — DDPG·SAC 공용"""
    for tp, sp in zip(target.parameters(), source.parameters()):
        tp.data.copy_(tau * sp.data + (1 - tau) * tp.data)

## 2교시 · DDPG 구현

`10:30 ~ 11:30` · `ddpg_pendulum.py`

> ⏳ 실행에 약 6분 0초 걸립니다 (학습 루프 — 멈춘 것이 아닙니다)

- Pendulum-v1(연속 행동)에서 DDPG 학습 루프를 완성한다
- 소프트 업데이트와 탐험 노이즈의 효과를 확인한다


In [ ]:
import gymnasium as gym
import torch
import torch.nn as nn
import numpy as np
import copy

env = gym.make("Pendulum-v1")
state_dim = env.observation_space.shape[0]   # 3
action_dim = env.action_space.shape[0]       # 1
max_action = float(env.action_space.high[0]) # 2.0

actor = Actor(state_dim, action_dim, max_action)
critic = Critic(state_dim, action_dim)
actor_t, critic_t = copy.deepcopy(actor), copy.deepcopy(critic)
actor_opt = torch.optim.Adam(actor.parameters(), lr=1e-4)
critic_opt = torch.optim.Adam(critic.parameters(), lr=1e-3)
buffer = ReplayBuffer(100_000, action_dtype=torch.float32)   # 연속 행동이므로 float32
gamma, batch_size, noise_std = 0.99, 128, 0.1

def train_step():
    s, a, r, s_next, done = buffer.sample(batch_size)
    # ── Critic 업데이트: TD 목표는 타깃 네트워크들로 ──
    with torch.no_grad():
        target_q = critic_t(s_next, actor_t(s_next))
        y = r + gamma * target_q * (1 - done)
    critic_loss = nn.functional.mse_loss(critic(s, a), y)
    critic_opt.zero_grad(); critic_loss.backward(); critic_opt.step()

    # ── Actor 업데이트: Q(s, μ(s))를 최대화 ──
    actor_loss = -critic(s, actor(s)).mean()
    actor_opt.zero_grad(); actor_loss.backward(); actor_opt.step()

    soft_update(actor_t, actor); soft_update(critic_t, critic)

returns = []
for episode in range(200):
    s, _ = env.reset()
    total, done = 0.0, False
    while not done:
        with torch.no_grad():
            a = actor(torch.as_tensor(s, dtype=torch.float32)).numpy()
        a += np.random.normal(0, noise_std * max_action, action_dim)   # 탐험 노이즈
        a = a.clip(-max_action, max_action)
        s_next, r, term, trunc, _ = env.step(a)
        done = term or trunc
        buffer.push(s, a, r, s_next, float(term))
        s, total = s_next, total + r
        if len(buffer) >= 1000:
            train_step()
    returns.append(total)
    if episode % 10 == 0:
        print(f"ep {episode:3d}  최근 10ep 평균 리턴 {np.mean(returns[-10:]):7.1f}")
# -1500 근처에서 시작해 -200 근처까지 오르면 성공

## 3교시 · Maximum Entropy RL 소개

`11:30 ~ 12:30` · `entropy_intuition.py`

- 최대 엔트로피 목적함수가 표준 RL 목적함수와 어떻게 다른지 이해한다
- 엔트로피 보너스가 탐험·강건성에 주는 효과를 설명할 수 있다


In [ ]:
import torch
import torch.nn.functional as F

# 엔트로피가 정책 분포에 주는 효과 직관 잡기
q_values = torch.tensor([1.0, 0.9, 0.2, -1.0])   # 행동 4개의 Q값

for alpha in [0.01, 0.5, 5.0]:
    # 최대 엔트로피 최적 정책: softmax(Q/alpha)
    pi = F.softmax(q_values / alpha, dim=0)
    entropy = -(pi * pi.log()).sum()
    print(f"alpha={alpha:4.2f}  pi={pi.numpy().round(3)}  H={entropy:.3f}")

# alpha=0.01 → 거의 greedy (첫 행동에 몰빵)
# alpha=0.5  → Q가 비슷한 행동 1,2를 골고루 선택  ← 여러 해를 포착
# alpha=5.0  → 거의 균등분포 (탐험 극대화)

## 4교시 · SAC 소개

`13:30 ~ 14:30` · `sac_actor.py`

- SAC의 3가지 핵심 구성요소(확률적 Actor, 트윈 Q, 자동 온도조절)를 이해한다
- 재매개변수화 트릭이 왜 필요한지 설명할 수 있다


In [ ]:
import torch
import torch.nn as nn

LOG_STD_MIN, LOG_STD_MAX = -20, 2

class GaussianActor(nn.Module):
    """SAC의 확률적 정책: tanh-squashed Gaussian"""
    def __init__(self, state_dim, action_dim, max_action):
        super().__init__()
        self.body = nn.Sequential(
            nn.Linear(state_dim, 256), nn.ReLU(),
            nn.Linear(256, 256), nn.ReLU(),
        )
        self.mu_head = nn.Linear(256, action_dim)
        self.log_std_head = nn.Linear(256, action_dim)
        self.max_action = max_action

    def forward(self, s):
        h = self.body(s)
        mu = self.mu_head(h)
        log_std = self.log_std_head(h).clamp(LOG_STD_MIN, LOG_STD_MAX)
        dist = torch.distributions.Normal(mu, log_std.exp())

        u = dist.rsample()              # 재매개변수화 샘플 (기울기 통과!)
        a = torch.tanh(u)
        # tanh 변환에 따른 log-prob 보정 (change of variables)
        log_prob = dist.log_prob(u).sum(-1)
        log_prob -= torch.log(1 - a.pow(2) + 1e-6).sum(-1)
        return a * self.max_action, log_prob

## 5교시 · TAC 소개

`14:30 ~ 15:30` · `q_log_intuition.py`

- 샤논 엔트로피의 한계와 Tsallis 엔트로피의 일반화를 이해한다
- TAC의 q 파라미터가 탐험 스타일에 주는 영향을 설명할 수 있다


In [ ]:
import torch

def q_log(x, q):
    """Tsallis q-logarithm: q→1이면 자연로그로 수렴"""
    if abs(q - 1.0) < 1e-6:
        return torch.log(x)
    return (x.pow(q - 1) - 1) / (q - 1)

# 같은 Q값에 대해 q에 따라 정책 분포가 어떻게 달라지는가 (수치 근사)
q_values = torch.tensor([1.0, 0.9, 0.2, -1.0])
alpha = 0.5

for q in [1.0, 1.5, 2.0]:
    # 정책 최적화를 경사하강으로 근사: max E[Q] + alpha * S_q(pi)
    logits = torch.zeros(4, requires_grad=True)
    opt = torch.optim.Adam([logits], lr=0.05)
    for _ in range(2000):
        pi = torch.softmax(logits, dim=0)
        entropy_q = -(pi * q_log(pi, q)).sum()
        loss = -((pi * q_values).sum() + alpha * entropy_q)
        opt.zero_grad(); loss.backward(); opt.step()
    print(f"q={q:.1f}  pi={pi.detach().numpy().round(3)}")

# q=1.0 → 모든 행동에 확률 배분 (SAC와 동일)
# q=2.0 → 나쁜 행동(Q=-1)의 확률이 사실상 0으로 — sparse한 탐험

## 6교시 · SAC 구현

`15:30 ~ 16:30` · `sac_pendulum.py`

> ⏳ 실행에 약 8분 43초 걸립니다 (학습 루프 — 멈춘 것이 아닙니다)

- Pendulum-v1에서 자동 온도조절을 포함한 SAC를 완성한다
- DDPG 대비 학습 안정성을 비교한다


In [ ]:
import gymnasium as gym
import torch
import torch.nn as nn
import numpy as np
import copy

env = gym.make("Pendulum-v1")
state_dim, action_dim = 3, 1
max_action = float(env.action_space.high[0])

actor = GaussianActor(state_dim, action_dim, max_action)
q1, q2 = Critic(state_dim, action_dim), Critic(state_dim, action_dim)   # 트윈 Q
q1_t, q2_t = copy.deepcopy(q1), copy.deepcopy(q2)
actor_opt = torch.optim.Adam(actor.parameters(), lr=3e-4)
q_opt = torch.optim.Adam(list(q1.parameters()) + list(q2.parameters()), lr=3e-4)

# 자동 온도 조절: log_alpha를 학습, 목표 엔트로피 = -action_dim
log_alpha = torch.zeros(1, requires_grad=True)
alpha_opt = torch.optim.Adam([log_alpha], lr=3e-4)
target_entropy = -action_dim

buffer = ReplayBuffer(100_000, action_dtype=torch.float32)   # 연속 행동이므로 float32
gamma, batch_size = 0.99, 256

def train_step():
    s, a, r, s_next, done = buffer.sample(batch_size)
    alpha = log_alpha.exp().detach()

    # ── 트윈 Q 업데이트: min(Q1',Q2') − α·logπ 로 soft TD 목표 ──
    with torch.no_grad():
        a_next, logp_next = actor(s_next)
        q_next = torch.min(q1_t(s_next, a_next), q2_t(s_next, a_next))
        y = r + gamma * (1 - done) * (q_next - alpha * logp_next)
    q_loss = nn.functional.mse_loss(q1(s, a), y) + nn.functional.mse_loss(q2(s, a), y)
    q_opt.zero_grad(); q_loss.backward(); q_opt.step()

    # ── Actor 업데이트: E[α·logπ − min(Q1,Q2)] 최소화 ──
    a_new, logp = actor(s)
    q_new = torch.min(q1(s, a_new), q2(s, a_new))
    actor_loss = (alpha * logp - q_new).mean()
    actor_opt.zero_grad(); actor_loss.backward(); actor_opt.step()

    # ── 온도 α 업데이트: 엔트로피를 목표치로 유지 ──
    alpha_loss = -(log_alpha.exp() * (logp + target_entropy).detach()).mean()
    alpha_opt.zero_grad(); alpha_loss.backward(); alpha_opt.step()

    soft_update(q1_t, q1); soft_update(q2_t, q2)

returns = []
for episode in range(150):
    s, _ = env.reset()
    total, done = 0.0, False
    while not done:
        with torch.no_grad():
            a, _ = actor(torch.as_tensor(s, dtype=torch.float32))
        s_next, r, term, trunc, _ = env.step(a.numpy())
        done = term or trunc
        buffer.push(s, a.numpy(), r, s_next, float(term))
        s, total = s_next, total + r
        if len(buffer) >= 1000:
            train_step()
    returns.append(total)
    if episode % 10 == 0:
        print(f"ep {episode:3d}  평균 {np.mean(returns[-10:]):7.1f}  alpha {log_alpha.exp().item():.3f}")

## 7교시 · TAC 구현

`16:30 ~ 17:30` · `tac_pendulum.py`

- SAC 코드에서 엔트로피 항을 Tsallis q-log로 교체해 TAC를 완성한다
- q 값을 바꿔가며 탐험 스타일의 변화를 관찰한다


In [ ]:
import torch

# SAC → TAC: 엔트로피 항의 log를 q-log로 교체하는 것이 전부
ENTROPIC_INDEX = 2.0        # q=1.0이면 SAC와 동일

def q_log_prob(log_prob, q=ENTROPIC_INDEX):
    """log π → log_q π 변환 (log_prob은 SAC actor가 주는 값)"""
    if abs(q - 1.0) < 1e-6:
        return log_prob                       # q→1: SAC로 환원
    prob = log_prob.exp().clamp(min=1e-8)
    return (prob.pow(q - 1) - 1) / (q - 1)    # q-logarithm

# ── SAC train_step에서 딱 두 곳만 수정 ──

# ① soft TD 목표 (수정 전: q_next - alpha * logp_next)
#    y = r + gamma * (1 - done) * (q_next - alpha * q_log_prob(logp_next))

# ② Actor 손실 (수정 전: alpha * logp - q_new)
#    actor_loss = (alpha * q_log_prob(logp) - q_new).mean()

# 실험 과제:
#  1. ENTROPIC_INDEX = 1.0으로 SAC와 동일 결과가 나오는지 검증
#  2. q = 1.5, 2.0에서 학습 곡선 비교
#  3. Pendulum은 행동공간이 작아 차이가 작습니다 —
#     HalfCheetah 등 고차원 환경에서 q>1의 효과가 뚜렷해집니다

# 3일 전체 요약
# Day1: MDP·벨만 → DP(모델有) → MC/TD(모델無) → SARSA/Q-Learning
# Day2: Q테이블 → 신경망(DQN·DDQN) / 정책 직접 학습(PG → A2C)
# Day3: 연속 행동(DDPG) → 최대 엔트로피(SAC) → 일반화 엔트로피(TAC)